# DDIM baseline + covariate CFG (TCGA, L1000 landmark genes)

Branch `cfg-demographics`. Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.

* **baseline** – tissue-conditioned DDIM (Lacan et al. replication, original label scheme)
* **cfg** – cancer type + sex + race + continuous age, with classifier-free guidance

Checkpoints and logs are written to Google Drive (`MyDrive/genaiproject_runs/<model>/`), so a dropped
session loses at most ~2000 steps: reconnect, run cells 1–3, then cell 4 with `RESUME = True`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive (checkpoints live here)
from google.colab import drive
drive.mount('/content/drive')
EXP_ROOT = '/content/drive/MyDrive/genaiproject_runs'
!mkdir -p {EXP_ROOT}

In [ ]:
# 3. Get the code (clone once, pull on later sessions) + the one missing dependency
import os
if not os.path.exists('/content/genaiproject'):
    !git clone -q -b cfg-demographics https://github.com/mzsanborn/genaiproject.git /content/genaiproject
else:
    !git -C /content/genaiproject pull -q
%cd /content/genaiproject
!git log --oneline -1
!pip install -q pytorch_warmup

## 4. Launch training

Runs in the background so the notebook stays usable; output goes to `EXP_ROOT/<model>.log`.
`WHICH = 'both'` trains the baseline, then the CFG model.

⚠️ With `RESUME = False` an existing run folder for that model is **deleted** and training restarts.
After a disconnect, set `RESUME = True`.

In [ ]:
WHICH = 'both'     # 'baseline', 'cfg' or 'both'
RESUME = False

models = ['baseline', 'cfg'] if WHICH == 'both' else [WHICH]
env = f"EXP_ROOT={EXP_ROOT} RESUME={int(RESUME)}"
cmd = ' && '.join(f"{env} bash run_colab.sh {m} > {EXP_ROOT}/{m}.log 2>&1" for m in models)
!nohup bash -c "{cmd}" > /dev/null 2>&1 &
print('launched:', models)

In [ ]:
# 5. Progress (re-run this cell whenever)
import glob, os, re, subprocess
for m in ['baseline', 'cfg']:
    log = f'{EXP_ROOT}/{m}.log'
    if not os.path.exists(log):
        print(f'{m}: not started'); continue
    lines = open(log, errors='ignore').read().splitlines()
    step = [l for l in lines if 'epoch:' in l and 'step:' in l]
    errs = [l for l in lines if 'Traceback' in l or 'Error' in l]
    print(f'--- {m}')
    if step: print('  ', re.sub(r'^.*? - (epoch:)', r'', step[-1]))
    print('   last line:', lines[-1][:200] if lines else '')
    if errs: print('   ERRORS:', *errs[-3:], sep='
   ')
print()
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# 6. Results once a model has finished (5 sampling runs: mean, std)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, m in enumerate(['baseline', 'cfg']):
    d = f'{EXP_ROOT}/{m}/logs/tcga'
    try:
        res = np.load(f'{d}/dict_res_metrics_5runs.npy', allow_pickle=True).item()
        print(m, {k: [round(float(x), 4) for x in v] for k, v in res.items()})
    except FileNotFoundError:
        print(m, ': no final metrics yet')
    try:
        loss = pd.read_csv(f'{d}/loss.csv').drop_duplicates('epoch', keep='last')
        ax[i].plot(loss['epoch'], loss['loss']); ax[i].set_title(f'{m} training loss'); ax[i].set_xlabel('epoch')
    except FileNotFoundError:
        pass
plt.tight_layout()

Notes
* Fréchet distance here is computed in a PCA space fitted on real training data (the original authors'
  pretrained classifier weights are not in the repo), so it is not numerically comparable to the paper.
* Kill a run: `!pkill -f main_ddim.py`